In [0]:
%run ../../common/_utils_config

In [0]:
%run ../../common/_utils_reference_data

In [0]:
%run ../utils/_utils_silver_contracts

In [0]:
%run ../utils/_utils_silver_mappings

In [0]:
import copy

failures = []


def check(description, condition):
    print(("✅ " if condition else "❌ ") + description)
    if not condition:
        failures.append(description)


def has_error(errors, text):
    return any(text in e for e in errors)


contracts = load_all_entity_contracts()
mappings = load_all_mappings()
print("Mappings found:", list(mappings))

mapping_errors = validate_all_mappings(contracts, mappings)
for name, errors in mapping_errors.items():
    print(f"   {name}: {errors}")

check("expected mappings exist",
      {"ree__electricity_price", "open_meteo_archive__weather_observation_hourly"} <= set(mappings))
check("all mappings in silver_specs are valid", not mapping_errors)

if mapping_errors:
    raise AssertionError("Fix the mappings above before running the next cells")

for mapping in mappings.values():
    print()
    print(describe_mapping(mapping, contracts))

In [0]:
def broken(mapping_name, change):
    """Validates a modified copy of a mapping."""
    mapping = copy.deepcopy(mappings[mapping_name])
    change(mapping)
    return validate_mapping(mapping, contracts)


ree = "ree__electricity_price"
weather = "open_meteo_archive__weather_observation_hourly"
resolution_by_code = {"from": "indicator.id", "map_values": {"1001": 60, "600": 15}}

cases = [
    ("wrong mapping name", ree,
     lambda m: m.update(mapping="ree__prices"), "mapping must be"),
    ("entity without contract", ree,
     lambda m: m.update(entity="unknown_entity"), "has no contract"),
    ("mapping into a reference entity", ree,
     lambda m: m.update(entity="location"), "mappings only apply to entities"),
    ("unknown bronze source", ree,
     lambda m: m["source"].update(source_name="unknown_source"), "has no bronze source spec"),
    ("unknown bronze endpoint", ree,
     lambda m: m["source"].update(endpoint="missing"), "has no endpoint"),
    ("parse step with an unknown alias", ree,
     lambda m: m["parse"][1].update(explode="item.attributes.values"), "unknown alias 'item'"),
    ("duplicated alias", ree,
     lambda m: m["parse"][1].update(**{"as": "indicator"}), "duplicated alias"),
    ("position on an endpoint without locations", ree,
     lambda m: m["parse"][0].update(position="row_index"), "locations block"),
    ("missing contract column", ree,
     lambda m: m["columns"].pop("price_eur_mwh"), "must declare every contract column"),
    ("column that is not in the contract", ree,
     lambda m: m["columns"].update(price_eur_kwh={"from": "point.value"}), "are not columns of"),
    ("from and constant together", ree,
     lambda m: m["columns"]["price_eur_mwh"].update(constant=1.0), "exactly one of from"),
    ("null business key", ree,
     lambda m: m["columns"].update(price_type={"constant": None}), "cannot be null"),
    ("map_values outside accepted_values", ree,
     lambda m: m["columns"]["price_type"].update(map_values={"1001": "pvpc", "600": "spot"}), "not in accepted_values"),
    ("map_values with unquoted numeric codes", ree,
     lambda m: m["columns"]["price_type"].update(map_values={1001: "pvpc"}), "quote numeric codes"),
    ("numeric map_values target outside the range", ree,
     lambda m: m["columns"].update(resolution_minutes={"from": "indicator.id", "map_values": {"1001": 60, "600": 2000}}),
     "outside the range"),
    ("text map_values target for a numeric column", ree,
     lambda m: m["columns"].update(resolution_minutes={"from": "indicator.id", "map_values": {"1001": "sixty"}}),
     "must be numbers"),
    ("timestamp without timezone", ree,
     lambda m: m["columns"]["datetime_utc"].pop("timezone"), "need timezone"),
    ("invalid timezone", ree,
     lambda m: m["columns"]["datetime_utc"].update(timezone="+01:00"), "timezone must be"),
    ("timezone on a numeric column", ree,
     lambda m: m["columns"]["price_eur_mwh"].update(timezone="UTC"), "timezone only applies to timestamp"),
    ("multiply on a string column", ree,
     lambda m: m["columns"]["price_type"].update(multiply=2), "multiply only applies to numeric"),
    ("location_code without position", ree,
     lambda m: m["columns"].update(price_type={"from": "location_code"}), "requires a parse step with position"),
    ("field that is not a zip_explode array", weather,
     lambda m: m["columns"]["temperature_c"].update(**{"from": "hour.temperature_80m"}),
     "one of the zip_explode arrays"),
    ("interval_of on a non-integer column", ree,
     lambda m: m["columns"].update(price_eur_mwh={"interval_of": "datetime_utc"}), "only applies to int or bigint"),
    ("interval_of pointing to a non-timestamp column", ree,
     lambda m: m["columns"].update(resolution_minutes={"interval_of": "price_type"}),
     "must name a timestamp business key"),
    ("interval_of combined with from", ree,
     lambda m: m["columns"].update(resolution_minutes={"interval_of": "datetime_utc", "from": "indicator.id"}),
     "exactly one of from"),
]

for description, mapping_name, change, expected in cases:
    errors = broken(mapping_name, change)
    check(f"rejects {description}", has_error(errors, expected))
    if not has_error(errors, expected):
        print(f"   got: {errors}")

# Valid variations
check("accepts 'constant: null' for a nullable column",
      not broken(weather, lambda m: m["columns"].update(temperature_c={"constant": None})))
check("accepts an IANA timezone",
      not broken(ree, lambda m: m["columns"]["datetime_utc"].update(timezone="Europe/Madrid")))
check("accepts a unit conversion",
      not broken(weather, lambda m: m["columns"]["wind_speed_10m_kmh"].update(multiply=3.6)))
check("accepts numeric map_values for an int column",
      not broken(ree, lambda m: m["columns"].update(resolution_minutes=resolution_by_code)))

In [0]:
if failures:
    raise AssertionError(f"{len(failures)} check(s) failed: {failures}")
print("✅ All silver mapping checks passed")